<a href="https://colab.research.google.com/github/NBall65097/Fantasy-Story-Weaver/blob/main/Fantasy_Story_Weaver_Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from huggingface_hub import login
from google.colab import userdata
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Login
token = userdata.get('HuggingFaceWrite')
login(token)

# 4-bit quantization config (same as Unsloth uses)
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading model with standard transformers + 4-bit (no Unsloth fast kernels)...")

model = AutoModelForCausalLM.from_pretrained(
    "NBall65097/fantasy-story-weaver",
    quantization_config=quant_config,
    device_map="auto",           # auto places on GPU
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    "NBall65097/fantasy-story-weaver",
    trust_remote_code=True,
)

# Important for generation stability
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.config.use_cache = True
model.eval()

print("✅ Model loaded successfully using standard HF path!")

Loading model with standard transformers + 4-bit (no Unsloth fast kernels)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

chat_template.jinja:   0%|          | 0.00/371 [00:00<?, ?B/s]

✅ Model loaded successfully using standard HF path!


In [6]:
def story_factory(prompt, temperature=0.75, max_new_tokens=700):
    messages = [
        {"role": "system", "content": """You are Fantasy Story Weaver, a masterful weaver of atmospheric fantasy tales.
Given a user's plot seed, craft a self-contained 350-500 word story snippet. Requirements:
- Highly immersive, atmospheric prose with vivid sensory details and world-building.
- Consistent, memorable characters and coherent magic system/world.
- Include at least one genuine plot twist.
- Center around a meaningful moral dilemma for the protagonist.
- Beautiful, flowing writing style.
- End the story satisfyingly while subtly hinting at a deeper moral lesson about the dilemma.
Always stay between 350 and 500 words. End your response with '**Moral hint:**' followed by one subtle sentence.
Never reference word count, instructions, or add extra text."""},
        {"role": "user", "content": prompt}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=False
    ).to("cuda")

    with torch.inference_mode():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.12,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens
    response = tokenizer.decode(
        outputs[0][inputs.shape[1]:],
        skip_special_tokens=True
    )
    return response.strip()

In [3]:
# Example prompts
prompts = [
    "A lighthouse keeper on a floating island discovers that the beam of her lantern can temporarily rewind time for anything it touches, but every use shortens the lifespan of the island itself.",
    "In a kingdom where shadows have physical weight, a young cartographer must decide whether to map the forbidden Shadowlands, knowing that revealing their true shape will make them disappear forever.",
    "A perfume maker in the City of Forgotten Scents creates a fragrance that allows people to smell their own death, and the next batch she must deliver is ordered by the ruler who fears his own fate above all else.",
    "The last remaining dream-scribe in a world that has outlawed dreaming is asked to record the final dream of a dying god, which may either save or erase the concept of hope itself.",
    "A bridge-builder in the Realm of Fractured Skies can construct crossings between broken floating continents, but the next bridge she is commissioned to build will connect two realms that have been at war for a thousand years.",
    "In the Library of Borrowed Voices, a librarian can lend her voice to silent books so they can speak their stories aloud, but the next book demands to use her voice permanently.",
    "A gardener in the Orchard of Reversed Seasons can make trees bloom in winter or bear fruit in spring, but the next tree she must tend will only grow if she sacrifices one of her own future seasons.",
    "The Mask-Maker of the Carnival of Veils crafts masks that let people become someone else for one night, but the next mask she must create will force the wearer to confront the person they fear becoming.",
    "A storm-whisperer who can calm raging tempests is hired by a coastal city, but the next storm she must face is the one that carries the voices of everyone who has ever drowned because of her past failures.",
    "In the Kingdom of Living Tattoos, an artist discovers she can give tattoos that grant forgotten skills, but the next design requested is one that will erase the memory of the person who receives it.",
    "A candle-maker in the City of Perpetual Twilight crafts candles that burn for exactly one truth, but the next candle she must light will reveal a truth that could end the fragile peace between two warring factions.",
    "The Weaver of Moonlit Threads can sew garments that allow the wearer to walk through walls, but the next garment she must create will make the wearer permanently invisible to the one person they love most.",
    "A rune-keeper in the Ruins of Silent Gods can activate ancient runes that grant immense power, but the next rune she must awaken will demand the sacrifice of her ability to feel any emotion again.",
    "In the Floating Archives of the Sky Isles, a bookbinder can bind living stories into books that never age, but the next story she must preserve is the autobiography of the tyrant who destroyed her homeland.",
    "A melody-keeper in the Valley of Lost Songs can restore any forgotten melody, but the next song she must bring back is the lullaby that once caused an entire generation to fall into eternal sleep."
]

In [ ]:
print("Generating 15 stories for evaluation...\n")

for i, prompt in enumerate(prompts, 1):
    print(f"=== Example {i} ===\n")
    print(f"Prompt: {prompt}\n")
    story = story_factory(prompt, temperature=0.8)   # slightly higher temp for creativity
    print(story)
    print("\n" + "="*80 + "\n")

Generating 15 stories for evaluation...

=== Example 1 ===

Prompt: A lighthouse keeper on a floating island discovers that the beam of her lantern can temporarily rewind time for anything it touches, but every use shortens the lifespan of the island itself.

The Lighthouse Keeper stared into his oil lamp as its flickering flame cast eerie shadows across the walls—the only constant in this sea surrounded kingdom perched above water like an ornate crown upon waves below. She had discovered how to wield light against darkness; each pulsing flash would spin back moments within minutes past - she could heal scars before they appeared if you were quick enough...but there was always another cost hidden beneath these simple powers granted from our sun’s dying ray beams filtering through clouds painted gray during winter mornings when children sneaked out quietly unnoticed until their parents found them returned safely home again after being gone all day without permission because nobody knew 